# Stool Melatonin Analysis

Summary: Exploratory analysis of gut melatonin.

Author, date: Fannie Kerff, January-August 2025

Environment: qiime2-amplicon-2024.10 (saved in `environment.yml`)

## Setup

In [ ]:
# imports
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from functions_script import load_qza_artifact, load_tsv, scatterplot, scatterplot_age, plot_estimates_vertically

%matplotlib inline

In [ ]:
# paths
# IF ACCESS TO ALL STUDY DATA
path = "/cluster/work/bokulich/fkerff/GrumpyBiome-analysis"
meta_data_path = f"{path}/data-sensitive/meta_data"
seq_data_path = f"{path}/data-sensitive/processed_data"
figures_path = f"{path}/figures"

# ELSE (ONLY ACCESS TO PUBLIC DATA)
# meta_data_path = "../data/meta_data"
# figures_path = "../figures"

In [ ]:
# create color palettes
colors = [plt.cm.Spectral(i/float(6)) for i in range(7)]
timepoints_colors = [colors[0], colors[5], colors[6]]

## Load samples metadata

In [ ]:
# load samples metadata
md_samples = load_tsv(f"{meta_data_path}/md_samples.tsv")
md_samples = md_samples.sort_values(by=["timepoint", "infant_id"])

print(md_samples.shape)
md_samples.head()

In [ ]:
# load genus feature table
genus_ft = load_qza_artifact(f"{seq_data_path}/9-comp-3035/genus-feature-table.qza")

## Gut melatonin analysis

In [ ]:
# filter data for gut melatonin values
cols=["Melatonin in Stool (pg/g)", 
      'time_since_last_bowel_movement_in_h', 
      'prior_time_spent_awake_in_h', 
      'time_since_last_feeding_in_h']
data_melatonin = md_samples.dropna(subset=cols)

In [ ]:
# rename melatonin column
data_melatonin = data_melatonin.rename(columns={"Melatonin in Stool (pg/g)": "Melatonin"})

In [ ]:
# linear mixed effects model for gut melatonin
model = smf.mixedlm("Melatonin ~ time_since_last_bowel_movement_in_h + prior_time_spent_awake_in_h + time_since_last_feeding_in_h + age_days + sex",
                    data_melatonin, 
                    groups=data_melatonin["infant_id"]).fit()
print(model.summary())

In [ ]:
# scatterplots of different factor effects on gut melatonin levels
fig, axs = plt.subplots(1, 4, figsize=(16, 6), sharey=True)

scatterplot(data_melatonin, "age_days", "Melatonin", "timepoint", 
                    "", "Melatonin in stool (pg/g)", "Age (days)",
                    timepoints_colors, axs[0], loc_position = 'upper center')

scatterplot_age(data_melatonin, "time_since_last_bowel_movement_in_h", "Melatonin", "timepoint", 
                    "", "", "Time since the last bowel movement (h)",
                    timepoints_colors, axs[1], loc_position = None)

scatterplot_age(data_melatonin, "prior_time_spent_awake_in_h", "Melatonin", "timepoint", 
                    "", "", "Prior time spent awake (h)",
                    timepoints_colors, axs[2], loc_position = None)

scatterplot_age(data_melatonin, "time_since_last_feeding_in_h", "Melatonin", "timepoint", 
                    "", "", "Time since the last feeding (h)",
                    timepoints_colors, axs[3], loc_position = None)

plt.tight_layout()

plt.savefig(f"{figures_path}/melatonin_scatter.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# coefficient plot
models = [model]
titles = ["Melatonin in Stool (pg/g)"]
plot_estimates_vertically(models, titles, f"{figures_path}/mel_estimates.pdf")

## Gut genus associated with melatonin levels

In [ ]:
# from processed_data/10-diff-abund-3035/genus-ancombc_results-BH-0.1.qzv (and processed_data/10-diff-abund-3035/genus-ancombc2_visualization-BH-0.1.qzv) results summary:
# 1 positive association between a genus abundance (lfc) and melatonin levels (BH-corrected q<0.1):
# d__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__Burkholderiales;f__Sutterellaceae;g__Sutterella has melatonin_est=0.0245, SE=7.11*10^-3, q=0.0730 (p=0.00143)

# 6 associations between genera and age (BH-corrected q<0.1):
# d__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__Burkholderiales;f__Sutterellaceae;g__Sutterella
# d__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__Enterobacterales;f__Enterobacteriaceae;__
# d__Bacteria;p__Firmicutes;c__Clostridia;o__Clostridiales;f__Clostridiaceae;g__Clostridium_sensu_stricto_1
# d__Bacteria;p__Firmicutes;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae;g__[Ruminococcus]_gnavus_group
# d__Bacteria;p__Firmicutes;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae;g__Lachnoclostridium
# d__Bacteria;p__Firmicutes;c__Negativicutes;o__Acidaminococcales;f__Acidaminococcaceae;g__Phascolarctobacterium

# family level: no significant associations

## Volcano plot of the DA results

### Import DA results (on `qiime2-amplicon-2025.7`)

In [ ]:
# # NB these 2 cells need to be run on the environment qiime2-amplicon-2025.7
# import qiime2 as q2
# from q2_composition._format import ANCOMBC2SliceMapping
# import pandas as pd

# path = "/cluster/work/bokulich/fkerff/GrumpyBiome-analysis"
# seq_data_path = f"{path}/data-sensitive/processed_data"
# meta_data_path = f"{path}/data-sensitive/meta_data"

# artifact = q2.Artifact.load(f"{seq_data_path}/10-diff-abund-3035/genus-ancombc_results-BH-0.1.qza")
# DA_results = artifact.view(ANCOMBC2SliceMapping)

# # combine the melatonin dictionary keys into one DataFrame
# DA_results_df = pd.DataFrame({
#     'taxon': DA_results['lfc']['taxon'],
#     'lfc_melatonin': DA_results['lfc']['melatonin'],
#     'se_melatonin': DA_results['se']['melatonin'],
#     'p_melatonin': DA_results['p']['melatonin'],
#     'q_melatonin': DA_results['q']['melatonin'],
#     'is_diff_melatonin': DA_results['diff']['melatonin']
# })

# # save to a TSV file
# DA_results_df.to_csv(f"{meta_data_path}/microbiome_melatonin_results.tsv", sep='\t', index=False)

In [ ]:
# # combine the age_days dictionary keys into one DataFrame
# DA_results_age_df = pd.DataFrame({
#     'taxon': DA_results['lfc']['taxon'],
#     'lfc_age_days': DA_results['lfc']['age_days'],
#     'se_age_days': DA_results['se']['age_days'],
#     'p_age_days': DA_results['p']['age_days'],
#     'q_age_days': DA_results['q']['age_days'],
#     'is_diff_age_days': DA_results['diff']['age_days']
# })

# # save to a TSV file
# DA_results_age_df.to_csv(f"{meta_data_path}/microbiome_age_days_results.tsv", sep='\t', index=False)

### Volcano plots

This transforms tiny p-values into large positive numbers so the "significant" points rise to the top.

In [ ]:
# load DA results
df = pd.read_csv(f"{meta_data_path}/microbiome_melatonin_results.tsv", sep='\t')

# prepare data
df_plot = df.copy()
df_plot['neg_log10_p'] = -np.log10(df_plot['p_melatonin'].replace(0, 1e-10))

# define significance based on q-value threshold
q_threshold = 0.1
df_plot['is_q_sig'] = df_plot['q_melatonin'] < q_threshold

plt.figure(figsize=(8, 6))

# plot non-significant points
ns = df_plot[~df_plot['is_q_sig']]
plt.scatter(ns['lfc_melatonin'], ns['neg_log10_p'], 
            c='lightgrey', alpha=0.5, label=f'q >= {q_threshold}')

# plot significant points (q < 0.1) in red
sig = df_plot[df_plot['is_q_sig']]
plt.scatter(sig['lfc_melatonin'], sig['neg_log10_p'], 
            c='crimson', alpha=0.9, edgecolors='black', linewidth=0.5, label=f'q < {q_threshold}')

# annotate with genus name and statistics
for i, row in sig.iterrows():
    taxonomy_parts = row['taxon'].split(';')
    genus_parts = [part for part in taxonomy_parts if part.startswith('g__')]
    raw_label = genus_parts[0].replace('g__', '') if genus_parts else taxonomy_parts[-1].split('__')[-1]
    label = (f"$\mathit{{{raw_label}}}$\n"
                f"LFC: {row['lfc_melatonin']:.4f} ± {row['se_melatonin']:.5f}\n"
                f"p: {row['p_melatonin']:.5f}\n"
                f"q: {row['q_melatonin']:.4f}")
    plt.annotate(
        label, 
        (row['lfc_melatonin']-0.006, row['neg_log10_p']-0.5),
        fontsize=9,
        fontweight='normal',
        xytext=(5, 5), 
        textcoords='offset points',
        va='bottom',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.3, ec='none'))

# format axes and labels
plt.axhline(-np.log10(0.05), color='black', linestyle=':', linewidth=1, label='p = 0.05')
plt.axvline(0, color='black', linewidth=0.8)

plt.xlabel('Gut melatonin (log fold change)', fontsize=12)
plt.ylabel('$-\log_{10}(pvalue)$', fontsize=12)
plt.legend(loc='upper left')
plt.grid(True, linestyle=':', alpha=0.4)

plt.tight_layout()

# plot and save
plt.savefig(f"{figures_path}/melatonin_DA_plot.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# load DA results
df = pd.read_csv(f"{meta_data_path}/microbiome_age_days_results.tsv", sep='\t')

# prepare data
df_plot = df.copy()
df_plot['neg_log10_p'] = -np.log10(df_plot['p_age_days'].replace(0, 1e-10))

# define significance based on q-value threshold
q_threshold = 0.1
df_plot['is_q_sig'] = df_plot['q_age_days'] < q_threshold

plt.figure(figsize=(8, 6))

# plot non-significant points
ns = df_plot[~df_plot['is_q_sig']]
plt.scatter(ns['lfc_age_days'], ns['neg_log10_p'], 
            c='lightgrey', alpha=0.5, label=f'q >= {q_threshold}')

# plot significant points (q < 0.1) in red
sig = df_plot[df_plot['is_q_sig']]
plt.scatter(sig['lfc_age_days'], sig['neg_log10_p'], 
            c='crimson', alpha=0.9, edgecolors='black', linewidth=0.5, label=f'q < {q_threshold}')

# annotate with genus name and statistics
for i, row in sig.iterrows():
    taxonomy_parts = row['taxon'].split(';')
    genus_parts = [part for part in taxonomy_parts if part.startswith('g__')]
    raw_label = genus_parts[0].replace('g__', '') if genus_parts else taxonomy_parts[-1].split('__')[-1]
    raw_label = raw_label.replace('_', "  ")
    if raw_label =='':
        family_parts = [part for part in taxonomy_parts if part.startswith('f__')]
        raw_label = family_parts[0].replace('f__', '') if family_parts else taxonomy_parts[-1].split('__')[-1]
        raw_label = 'Unclassified ' + raw_label
    if raw_label == "Sutterella":
        label = (f"$\mathit{{{raw_label}}}$\n"
                f"LFC: {row['lfc_age_days']:.4f} ± {row['se_age_days']:.5f}\n"
                f"p: {row['p_age_days']:.5f}\n"
                f"q: {row['q_age_days']:.4f}")
        dist=0.2
    else:
        label = raw_label
        dist=-0.4
    plt.annotate(
        label, 
        (row['lfc_age_days']-0.014, row['neg_log10_p']+dist),
        fontsize=9,
        fontweight='normal',
        xytext=(5, 5), 
        textcoords='offset points',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.3, ec='none'))

# format axes and labels
plt.axhline(-np.log10(0.05), color='black', linestyle=':', linewidth=1, label='p = 0.05')
plt.axvline(0, color='black', linewidth=0.8)

plt.xlabel('Age in days (log fold change)', fontsize=12)
plt.ylabel('$-\log_{10}(pvalue)$', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, linestyle=':', alpha=0.4)

plt.tight_layout()

# plot and save
plt.savefig(f"{figures_path}/age_days_DA_plot.pdf", dpi=300, bbox_inches='tight')
plt.show()